In [ ]:
!git clone https://github.com/michelecafagna26/vinvl-visualbackbone.git

In [ ]:
!pip install --upgrade pip

In [ ]:
pip install -r /kaggle/working/vinvl-visualbackbone/requirements.txt

In [ ]:
%cd /kaggle/working/vinvl-visualbackbone/scene_graph_benchmark/

In [ ]:
import re

# List of CUDA files to patch
cuda_files = [
    "/kaggle/working/vinvl-visualbackbone/scene_graph_benchmark/maskrcnn_benchmark/csrc/cuda/ROIAlign_cuda.cu",
    "/kaggle/working/vinvl-visualbackbone/scene_graph_benchmark/maskrcnn_benchmark/csrc/cuda/ROIPool_cuda.cu",
    "/kaggle/working/vinvl-visualbackbone/scene_graph_benchmark/maskrcnn_benchmark/csrc/cuda/SigmoidFocalLoss_cuda.cu",
    "/kaggle/working/vinvl-visualbackbone/scene_graph_benchmark/maskrcnn_benchmark/csrc/cuda/deform_conv_cuda.cu",
    "/kaggle/working/vinvl-visualbackbone/scene_graph_benchmark/maskrcnn_benchmark/csrc/cuda/deform_conv_kernel_cuda.cu",
    "/kaggle/working/vinvl-visualbackbone/scene_graph_benchmark/maskrcnn_benchmark/csrc/cuda/deform_pool_cuda.cu",
    "/kaggle/working/vinvl-visualbackbone/scene_graph_benchmark/maskrcnn_benchmark/csrc/cuda/deform_pool_kernel_cuda.cu",
    "/kaggle/working/vinvl-visualbackbone/scene_graph_benchmark/maskrcnn_benchmark/csrc/cuda/nms.cu"
]

for cu_file_path in cuda_files:
    # Read original file
    with open(cu_file_path, "r", encoding="utf-8") as f:
        content = f.read()

    # 1️⃣ Remove THC includes
    content = re.sub(r'#include <THC/THC.*?>', '', content)

    # 2️⃣ Replace THCCeilDiv(a,b) with (a+b-1)/b
    content = re.sub(r'THCCeilDiv\(([^,]+),\s*([^)]+)\)', r'(\1 + \2 - 1)/\2', content)

    # 3️⃣ Replace THCudaCheck(...) with TORCH_CHECK
    content = re.sub(
        r'THCudaCheck\((.*?)\);', 
        r'TORCH_CHECK(\1 == cudaSuccess, "CUDA kernel failed");',
        content
    )

    # Save patched file
    with open(cu_file_path, "w", encoding="utf-8") as f:
        f.write(content)

    print(f"Patched {cu_file_path} successfully for Kaggle / modern PyTorch!")

In [ ]:
import re

deform_file = "/kaggle/working/vinvl-visualbackbone/scene_graph_benchmark/maskrcnn_benchmark/csrc/cuda/deform_conv_kernel_cuda.cu"

with open(deform_file, "r", encoding="utf-8") as f:
    code = f.read()

# Replace atomicAdd(...) for c10::Half
pattern = r'atomicAdd\(([^,]+), ([^\)]+)\);'
replacement = r'atomicAdd(reinterpret_cast<float*>(\1), static_cast<float>(\2));'

code = re.sub(pattern, replacement, code)

with open(deform_file, "w", encoding="utf-8") as f:
    f.write(code)

print("Patched deform_conv_kernel_cuda.cu for c10::Half atomicAdd")

In [ ]:
import os

nms_path = '/kaggle/working/vinvl-visualbackbone/scene_graph_benchmark/maskrcnn_benchmark/csrc/cuda/nms.cu'

patch_content = """
// Copyright (c) Facebook, Inc. and its affiliates. All Rights Reserved.
#include <ATen/ATen.h>
#include <ATen/cuda/CUDAContext.h>
#include <c10/cuda/CUDAException.h>
#include <c10/cuda/CUDAGuard.h>
#include <iostream>
#include <vector>

// Define the block size explicitly
#define threadsPerBlock 64
#define DIVUP(m,n) ((m) / (n) + ((m) % (n) > 0))

__device__ inline float devIoU(float const * const a, float const * const b) {
  float left = max(a[0], b[0]), right = min(a[2], b[2]);
  float top = max(a[1], b[1]), bottom = min(a[3], b[3]);
  float width = max(right - left, 0.f), height = max(bottom - top, 0.f);
  float interS = width * height;
  float Sa = (a[2] - a[0]) * (a[3] - a[1]);
  float Sb = (b[2] - b[0]) * (b[3] - b[1]);
  return interS / (Sa + Sb - interS);
}

__global__ void nms_kernel(const int n_boxes, const float nms_overlap_thresh,
                           const float *dev_boxes, unsigned long long *dev_mask) {
  const int row_start = blockIdx.y;
  const int col_start = blockIdx.x;

  const int row_size =
        min(n_boxes - row_start * threadsPerBlock, threadsPerBlock);
  const int col_size =
        min(n_boxes - col_start * threadsPerBlock, threadsPerBlock);

  __shared__ float block_boxes[threadsPerBlock * 4];
  if (threadIdx.x < col_size) {
    block_boxes[threadIdx.x * 4 + 0] =
        dev_boxes[(threadsPerBlock * col_start + threadIdx.x) * 4 + 0];
    block_boxes[threadIdx.x * 4 + 1] =
        dev_boxes[(threadsPerBlock * col_start + threadIdx.x) * 4 + 1];
    block_boxes[threadIdx.x * 4 + 2] =
        dev_boxes[(threadsPerBlock * col_start + threadIdx.x) * 4 + 2];
    block_boxes[threadIdx.x * 4 + 3] =
        dev_boxes[(threadsPerBlock * col_start + threadIdx.x) * 4 + 3];
  }
  __syncthreads();

  if (threadIdx.x < row_size) {
    const int cur_box_idx = threadsPerBlock * row_start + threadIdx.x;
    const float *cur_box = dev_boxes + cur_box_idx * 4;
    int i = 0;
    unsigned long long t = 0;
    int start = 0;
    if (row_start == col_start) {
      start = threadIdx.x + 1;
    }
    for (i = start; i < col_size; i++) {
      if (devIoU(cur_box, block_boxes + i * 4) > nms_overlap_thresh) {
        t |= 1ULL << i;
      }
    }
    const int col_blocks = DIVUP(n_boxes, threadsPerBlock);
    dev_mask[cur_box_idx * col_blocks + col_start] = t;
  }
}

at::Tensor nms_cuda(const at::Tensor boxes, float nms_overlap_thresh) {
  // Use c10::cuda namespace for CUDAGuard in modern Torch
  c10::cuda::CUDAGuard device_guard(boxes.device());

  auto boxes_num = boxes.size(0);
  const int col_blocks = DIVUP(boxes_num, threadsPerBlock);

  auto mask_options = boxes.options().dtype(at::kLong);
  at::Tensor mask = at::empty({boxes_num * col_blocks}, mask_options);

  dim3 blocks(col_blocks, col_blocks);
  dim3 threads(threadsPerBlock);

  nms_kernel<<<blocks, threads, 0, at::cuda::getCurrentCUDAStream()>>>(
      boxes_num,
      nms_overlap_thresh,
      boxes.data_ptr<float>(),
      (unsigned long long*)mask.data_ptr<int64_t>());

  C10_CUDA_CHECK(cudaGetLastError());
  return mask;
}
"""

with open(nms_path, 'w') as f:
    f.write(patch_content)

print(f"Patched nms.cu with threadsPerBlock and correct CUDAGuard namespace.")

In [ ]:
deform_path = '/kaggle/working/vinvl-visualbackbone/scene_graph_benchmark/maskrcnn_benchmark/csrc/cuda/deform_pool_kernel_cuda.cu'

with open(deform_path, 'r') as f:
    lines = f.readlines()

# Insert the atomic header if not present
if '#include <ATen/cuda/Atomic.cuh>\n' not in lines:
    lines.insert(0, '#include <ATen/cuda/Atomic.cuh>\n')

with open(deform_path, 'w') as f:
    f.writelines(lines)

print(f"Successfully patched: {deform_path}")

In [ ]:
import os

# Define paths
cuda_dir = '/kaggle/working/vinvl-visualbackbone/scene_graph_benchmark/maskrcnn_benchmark/csrc/cuda/'
duplicate_file = os.path.join(cuda_dir, 'nms_patched.cu')

# 1. Remove the duplicate source file
if os.path.exists(duplicate_file):
    os.remove(duplicate_file)
    print(f"Removed duplicate file: {duplicate_file}")

# 2. Clean the build directory to ensure no stale object files exist
# This is crucial to get rid of nms_patched.o
!rm -rf /kaggle/working/vinvl-visualbackbone/scene_graph_benchmark/build/

print("Cleaned up duplicates. Ready to rebuild.")

In [ ]:
!rm -rf build/
!python setup.py build develop

In [ ]:
!git clone https://huggingface.co/michelecafagna26/vinvl_vg_x152c4 scene_graph_benchmark/models/vinvl

In [ ]:
import os
import sys

sys.path.append('/kaggle/working/vinvl-visualbackbone/scene_graph_benchmark')

In [ ]:
import os

file_path = "/kaggle/working/vinvl-visualbackbone/scene_graph_benchmark/maskrcnn_benchmark/utils/imports.py"

with open(file_path, "r") as f:
    content = f.read()

# Replace torch._six usage
content = content.replace(
    "if torch._six.PY3:",
    "if True:"
)

with open(file_path, "w") as f:
    f.write(content)

print("✅ Patched torch._six issue!")

In [ ]:
file_path = "/kaggle/working/vinvl-visualbackbone/scene_graph_benchmark/maskrcnn_benchmark/utils/model_zoo.py"

new_content = """
# Patched model_zoo for modern PyTorch

import os
import sys
import hashlib
import requests
from urllib.parse import urlparse

def download_url_to_file(url, dst, hash_prefix=None, progress=True):
    response = requests.get(url, stream=True)
    response.raise_for_status()
    
    with open(dst, "wb") as f:
        for chunk in response.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)

def cache_url(url, model_dir=None, progress=True):
    if model_dir is None:
        model_dir = os.path.join(os.path.expanduser("~"), ".torch", "models")

    os.makedirs(model_dir, exist_ok=True)
    
    filename = os.path.basename(urlparse(url).path)
    cached_file = os.path.join(model_dir, filename)

    if not os.path.exists(cached_file):
        print(f"Downloading: {url} to {cached_file}")
        download_url_to_file(url, cached_file)

    return cached_file
"""

with open(file_path, "w") as f:
    f.write(new_content)

print("✅ Patched model_zoo.py!")

In [ ]:
import os

root_dir = "/kaggle/working/vinvl-visualbackbone"

for subdir, _, files in os.walk(root_dir):
    for file in files:
        if file.endswith(".py"):
            file_path = os.path.join(subdir, file)
            
            with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
                content = f.read()
            
            new_content = content.replace("np.float", "float")
            
            if new_content != content:
                with open(file_path, "w", encoding="utf-8") as f:
                    f.write(new_content)

print("✅ Replaced all np.float -> float")

In [ ]:
import numpy as np

np.float = float
np.int = int
np.bool = bool

print("✅ Monkey patched numpy aliases")

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import cv2
import json
from pathlib import Path
from clint.textui import progress
import requests
import h5py

from scene_graph_benchmark.AttrRCNN import AttrRCNN
from scene_graph_benchmark.scene_parser import SceneParser
from scene_graph_benchmark.config import sg_cfg
from maskrcnn_benchmark.config import cfg
from maskrcnn_benchmark.data.transforms import build_transforms
from maskrcnn_benchmark.utils.miscellaneous import set_seed
from maskrcnn_benchmark.utils.checkpoint import DetectronCheckpointer
from scene_graph_benchmark.wrappers.utils import cv2Img_to_Image


BASE_PATH = '/kaggle/working/vinvl-visualbackbone/scene_graph_benchmark/scene_graph_benchmark'
CONFIG_FILE = '/kaggle/working/vinvl-visualbackbone/scene_graph_benchmark/sgg_configs/vgattr/vinvl_x152c4.yaml'

MODEL_DIR = Path('/kaggle/working/vinvl-visualbackbone/scene_graph_benchmark/scene_graph_benchmark/models/vinvl')
_MODEL_URL = "https://penzhanwu2.blob.core.windows.net/sgg/sgg_benchmark/vinvl_model_zoo/vinvl_vg_x152c4.pth"
_LABEL_URL = "https://penzhanwu2.blob.core.windows.net/sgg/sgg_benchmark/vinvl_model_zoo/VG-SGG-dicts-vgoi6-clipped.json"


class VinVLCpuTopKExtractor(nn.Module):
    def __init__(self, config_file=None, opts=None):
        super().__init__()

        num_of_gpus = torch.cuda.device_count()
        set_seed(1000, num_of_gpus)

        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.k = 36

        self.opts = {
            "MODEL.WEIGHT": "models/vinvl/vinvl_vg_x152c4.pth",
            "MODEL.ROI_HEADS.NMS_FILTER": 1,
            "MODEL.ROI_HEADS.SCORE_THRESH": 0.2,
            "TEST.IGNORE_BOX_REGRESSION": True,
            "DATASETS.LABELMAP_FILE": "models/vinvl/VG-SGG-dicts-vgoi6-clipped.json",
            "TEST.OUTPUT_FEATURE": True
        }
        if opts:
            self.opts.update(opts)

        self.config_file = config_file if config_file else CONFIG_FILE

        cfg.set_new_allowed(True)
        cfg.merge_from_other_cfg(sg_cfg)
        cfg.merge_from_file(self.config_file)
        cfg.update(self.opts)
        cfg.set_new_allowed(False)
        cfg.freeze()

        if cfg.MODEL.META_ARCHITECTURE == "AttrRCNN":
            self.model = AttrRCNN(cfg)
        else:
            raise ValueError(
                f"{cfg.MODEL.META_ARCHITECTURE} is not a valid MODEL.META_ARCHITECTURE; it must be 'AttrRCNN'"
            )

        self.model.eval()
        self.model.to(self.device)

        weight_path = Path(BASE_PATH, cfg.MODEL.WEIGHT)
        label_path = Path(BASE_PATH, cfg.DATASETS.LABELMAP_FILE)

        if not weight_path.is_file():
            print(f"{cfg.MODEL.WEIGHT} not found")
            MODEL_DIR.mkdir(parents=True, exist_ok=True)
            print(f"created {MODEL_DIR}")

            print(f"downloading {Path(_MODEL_URL).name}")
            r = requests.get(_MODEL_URL, stream=True)
            path = Path(MODEL_DIR, Path(_MODEL_URL).name)
            with open(path, 'wb') as f:
                total_length = int(r.headers.get('content-length', 0))
                for chunk in progress.bar(r.iter_content(chunk_size=1024), expected_size=(total_length / 1024) + 1):
                    if chunk:
                        f.write(chunk)
                        f.flush()

            print(f"downloading {Path(_LABEL_URL).name}")
            r = requests.get(_LABEL_URL, stream=True)
            path = Path(MODEL_DIR, Path(_LABEL_URL).name)
            with open(path, 'wb') as f:
                total_length = int(r.headers.get('content-length', 0))
                for chunk in progress.bar(r.iter_content(chunk_size=1024), expected_size=(total_length / 1024) + 1):
                    if chunk:
                        f.write(chunk)
                        f.flush()

        self.checkpointer = DetectronCheckpointer(cfg, self.model, save_dir="")
        self.checkpointer.load(str(weight_path))

        with open(label_path, "rb") as fp:
            label_dict = json.load(fp)

        self.idx2label = {int(k): v for k, v in label_dict["idx_to_label"].items()}
        self.label2idx = {k: int(v) for k, v in label_dict["label_to_idx"].items()}
        self.idx2attr = {int(k): v for k, v in label_dict["idx_to_attribute"].items()}
        self.attr2idx = {k: int(v) for k, v in label_dict["attribute_to_idx"].items()}

        self.transforms = build_transforms(cfg, is_train=False)

    @torch.no_grad()
    def _extract_single_image(self, img):
        if isinstance(img, str):
            img = cv2.imread(img)
        if img is None:
            raise ValueError("Failed to read image.")

        img_input = cv2Img_to_Image(img)
        img_input, _ = self.transforms(img_input, target=None)
        img_input = img_input.to(self.device)

        prediction = self.model(img_input)
        prediction = prediction[0].to(self.device)

        img_h, img_w = img.shape[:2]
        prediction = prediction.resize((img_w, img_h))

        boxes = prediction.bbox                      # [num_boxes, 4]
        scores = prediction.get_field("scores")     # [num_boxes]
        classes = prediction.get_field("labels")    # [num_boxes]
        features = prediction.get_field("box_features")  # [num_boxes, 2048]

        # -------- Dynamic NMS --------
        selected_indices = None
        for nms_thresh in np.arange(0.5, 1.0, 0.1):
            keep = torch.ops.torchvision.nms(boxes, scores, float(nms_thresh))
            if len(keep) >= self.k:
                selected_indices = keep[:self.k]
                break

        # fallback: top-k by detector score
        if selected_indices is None or len(selected_indices) < self.k:
            topk = min(self.k, scores.numel())
            _, indices = torch.topk(scores, k=topk)
            selected_indices = indices

            if topk < self.k:
                pad_len = self.k - topk
                selected_indices = torch.cat([
                    selected_indices,
                    selected_indices[-1:].repeat(pad_len)
                ])

        selected_indices = selected_indices.to(boxes.device)

        selected_boxes = boxes[selected_indices].detach().cpu()          # [k, 4]
        selected_scores = scores[selected_indices].detach().cpu()        # [k]
        selected_classes = classes[selected_indices].detach().cpu()      # [k]
        selected_features = features[selected_indices].detach().cpu()    # [k, 2048]

        selected_label_names = [
            self.idx2label.get(int(c.item()), "unknown")
            for c in selected_classes
        ]

        return {
            "features": selected_features,                    # torch.FloatTensor [k, 2048]
            "boxes": selected_boxes,                          # torch.FloatTensor [k, 4]
            "scores": selected_scores,                        # torch.FloatTensor [k]
            "label_ids": selected_classes.long(),             # torch.LongTensor [k]
            "label_names": selected_label_names               # list[str] len=k
        }

    @torch.no_grad()
    def forward(self, *images):
        self.model.eval()

        batch_features = []
        batch_boxes = []
        batch_scores = []
        batch_label_ids = []
        batch_label_names = []

        for image in images:
            out = self._extract_single_image(image)

            batch_features.append(out["features"])
            batch_boxes.append(out["boxes"])
            batch_scores.append(out["scores"])
            batch_label_ids.append(out["label_ids"])
            batch_label_names.append(out["label_names"])

        batch_features = torch.stack(batch_features, dim=0)     # [B, k, 2048]
        batch_boxes = torch.stack(batch_boxes, dim=0)           # [B, k, 4]
        batch_scores = torch.stack(batch_scores, dim=0)         # [B, k]
        batch_label_ids = torch.stack(batch_label_ids, dim=0)   # [B, k]

        return {
            "features": batch_features,
            "boxes": batch_boxes,
            "scores": batch_scores,
            "label_ids": batch_label_ids,
            "label_names": batch_label_names
        }

In [ ]:
detector = VinVLCpuTopKExtractor()

In [ ]:
import argparse
base_path = "/kaggle/working/ViVQA-LanguageBias"
opt = argparse.Namespace()

# Training Parameters
opt.output_dir = "/kaggle/working/output"
opt.log_level = "passive"
opt.lr_scheduler_type = "cosine"
opt.warmup_ratio = 0.1
opt.logging_strategy = "epoch"
opt.save_strategy = "epoch"
opt.save_total_limit = 1
opt.train_batch_size = 16
opt.eval_batch_size = 16
opt.epochs = 2
opt.learning_rate = 3e-5
opt.weight_decay = 0.01
opt.workers = 2

# Dataset Paths (Absolute Kaggle Paths)
opt.image_path = f"/kaggle/input/datasets/luan85/viet-vqa/images"
opt.ans_path = f"/kaggle/input/datasets/luan85/viet-vqa/vocab.json"
opt.train_path = f"/kaggle/input/datasets/luan85/viet-vqa/train.csv"
opt.val_path = f"{base_path}/data/ViVQA-csv/val.csv"
opt.test_path = f"/kaggle/input/datasets/luan85/viet-vqa/test.csv"
opt.feature_paths = f"/kaggle/working/"

# Model Architecture
opt.drop_path_rate = 0.3
opt.encoder_layers = 6
opt.encoder_attention_heads_layers = 6
opt.classes = 353

In [ ]:
import torch
from torch.utils.data import DataLoader
import os
import glob
from tqdm import tqdm
import numpy as np
import h5py

def extract_features(vs_encoder, opt, model_name):
    file_paths = sorted(glob.glob(os.path.join(opt.image_path, '*.jpg')))
    num_images = len(file_paths)
    k = vs_encoder.k

    str_dtype = h5py.string_dtype(encoding='utf-8')

    with h5py.File(f'{opt.feature_paths}/{model_name}.hdf5', 'w') as f:
        features_ds = f.create_dataset('features', shape=(num_images, k, 2048), dtype='float16')
        boxes_ds = f.create_dataset('boxes', shape=(num_images, k, 4), dtype='float16')
        scores_ds = f.create_dataset('scores', shape=(num_images, k), dtype='float16')
        label_ids_ds = f.create_dataset('label_ids', shape=(num_images, k), dtype='int32')
        label_names_ds = f.create_dataset('label_names', shape=(num_images, k), dtype=str_dtype)
        img_ids_ds = f.create_dataset('ids', shape=(num_images,), dtype='int32')

        j = 0
        data_loader = DataLoader(file_paths, batch_size=32, shuffle=False, num_workers=4)

        for paths in tqdm(data_loader):
            ids = [int(os.path.basename(path).split('.')[0]) for path in paths]

            with torch.no_grad():
                out = vs_encoder(*paths)

            bsz = len(paths)
            k_end = j + bsz

            features_ds[j:k_end] = out["features"].cpu().numpy().astype('float16')
            boxes_ds[j:k_end] = out["boxes"].cpu().numpy().astype('float16')
            scores_ds[j:k_end] = out["scores"].cpu().numpy().astype('float16')
            label_ids_ds[j:k_end] = out["label_ids"].cpu().numpy().astype('int32')

            # label_names is list[list[str]]
            label_names_array = np.array(out["label_names"], dtype=object)
            label_names_ds[j:k_end] = label_names_array

            img_ids_ds[j:k_end] = np.array(ids, dtype='int32')

            j = k_end

    print(f"Saved to: {opt.feature_paths}/{model_name}.hdf5")

In [ ]:
extract_features(detector, opt, "VinVL")